In [ ]:
# === Pacotes necessários / Required packages ===
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# dados não indexados / unindexed data

In [ ]:
# === Ler os dados .CSV / Read .CSV data ===
dados = pd.read_csv('NAME.csv', sep=';')  # especifica o separador / specifies the separator
print(dados.head())

In [ ]:
# === Ler os dados .XLSX  / Read .XLSX data ===
dados = pd.read_excel("NAME.xlsx")
print(dados_com_clusters.head())

In [ ]:
# === Padronizar os dados / Standardize data ===
scaler = StandardScaler()
dados_padronizados = scaler.fit_transform(dados)

In [ ]:
# === K-means ===
np.random.seed(1234)
kmeans = KMeans(n_clusters=2, n_init=25, random_state=1234)
clusters = kmeans.fit_predict(dados_padronizados)

In [ ]:
# === PCA para reduzir a 2D para visualização / PCA to reduce to 2D for visualization ===
pca = PCA(n_components=2)
dados_pca = pca.fit_transform(dados_padronizados)

# === Gráfico dos clusters / Cluster graph ===
plt.figure(figsize=(8, 6))
scatter = plt.scatter(dados_pca[:, 0], dados_pca[:, 1], c=clusters, cmap="tab10", s=50, alpha=0.8)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-means Clustering - PCA 2D")
plt.legend(*scatter.legend_elements(), title="Cluster")
plt.show()

In [ ]:
# === Salvar resultados / Save results ===
dados_com_clusters = pd.DataFrame(dados_padronizados, columns=dados.columns)
dados_com_clusters.insert(0, "Cluster", clusters + 1)  # +1 para ficar no estilo do R (1, 2, ...)
dados_com_clusters.to_excel("dados_com_clusters.xlsx", index=False)

print(dados_com_clusters.head())

Para as variáveis observáveis / For observable variables

In [ ]:
# === Transposta do DataFrame / Transpose of the DataFrame ===
dados_transposto = dados.T
display(dados_transposto.head())

In [ ]:
# === K-means ===
np.random.seed(1234)
kmeans = KMeans(n_clusters=3, n_init='auto', random_state=1234) # n_clusters = number of classes
clusters = kmeans.fit_predict(dados_transposto)

In [ ]:
# === PCA para reduzir a 2D para visualização / PCA to reduce to 2D for visualization ===
# Usamos o dados_transposto completo de forma direta sem nenhuma exclusão ou faturamento
pca = PCA(n_components=2)
dados_pca = pca.fit_transform(dados_transposto)

# === Gráfico dos clusters / Cluster graph ===
plt.figure(figsize=(10, 8))
scatter = plt.scatter(dados_pca[:, 0], dados_pca[:, 1], c=clusters, cmap="tab10", s=100, alpha=0.8)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-means Clustering (Features) - PCA 2D")

# Adicionar o nome de cada amostra (atributo) ao lado de seu ponto correspondente
for i, nome_amostra in enumerate(dados_transposto.index):
    plt.annotate(nome_amostra, (dados_pca[i, 0], dados_pca[i, 1]), textcoords="offset points", xytext=(5,5), ha='left', fontsize=9, fontweight='bold')

# Criando uma legenda simples para os clusters gerados
handles, labels = scatter.legend_elements()
plt.legend(handles, [f'Cluster {i}' for i in range(len(handles))], title="Clusters", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# dados indexados / indexed data

In [ ]:
# === Ler os dados .CSV / Read .CSV data ===
dados = pd.read_csv('NAME.csv', sep=',')  # especifica o separador / specifies the separator
print(dados.head())

In [ ]:
# === Ler os dados .XLSX / Read .XSLX data ===
dados = pd.read_excel("NAME.xlsx")
print(dados.head())

In [ ]:
# Se a primeira coluna for a classe, separa / If the first column is the class, separate
if dados.iloc[:, 0].dtype == object or dados.iloc[:, 0].dtype == int:
    classes = dados.iloc[:, 0]  # guarda as classes originais / saves the original classes
    dados = dados.iloc[:, 1:]   # usa só as variáveis numéricas / uses only numeric variables

# Identificar e contar as classes únicas / identify and count the unique classes
class_counts = classes.value_counts()

print("Class names and their quantities:")
print(class_counts)
print(f"Total number of classes: {len(class_counts)}")

# === Padronizar os dados / Standardize data ===
scaler = StandardScaler()
dados_padronizados = scaler.fit_transform(dados)

In [ ]:
# === K-means ===
np.random.seed(1234)
kmeans = KMeans(n_clusters=len(class_counts), n_init='auto', random_state=1234) # n_clusters = number of classes
clusters = kmeans.fit_predict(dados_padronizados)

In [ ]:
# === PCA para reduzir a 2D para visualização / PCA to reduce to 2D for visualization ===
pca = PCA(n_components=2)
dados_pca = pca.fit_transform(dados_padronizados)

# === Gráfico dos clusters / Cluster graph ===
plt.figure(figsize=(8, 6))
scatter = plt.scatter(dados_pca[:, 0], dados_pca[:, 1], c=clusters, cmap="tab10", s=50, alpha=0.8)
plt.xlabel("Atribute 1")
plt.ylabel("Atribute 2")
plt.title("K-means Clustering")

# Create a mapping from cluster ID to the most frequent true class label in that cluster
cluster_to_class_map = {}
for i in np.unique(clusters):
    # Get true class labels for points in the current cluster
    true_labels_in_cluster = classes[clusters == i]
    # Find the most frequent true class label
    most_frequent_class = true_labels_in_cluster.mode()[0]
    cluster_to_class_map[i] = most_frequent_class

# Create custom labels for the legend based on the mapping
legend_labels = [cluster_to_class_map[i] for i in sorted(cluster_to_class_map.keys())]

# Get the legend handles from the scatter plot
handles, _ = scatter.legend_elements()

# Position the legend outside the plot and use custom labels
plt.legend(handles, legend_labels, title="Class", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout() # Adjust layout to prevent legend from overlapping
plt.show()

In [ ]:
#Criar um DataFrame com a classe original e os dados PCA
dados_para_salvar = pd.DataFrame({
    'Classe Original': classes,
    'PC1': dados_pca[:, 0],
    'PC2': dados_pca[:, 1]
})

# Salvar o DataFrame em um arquivo Excel
dados_para_salvar.to_excel('clusters_pca_data.xlsx', index=False)

print("Arquivo 'clusters_pca_data.xlsx' salvo com sucesso!")
print(dados_para_salvar.head())

In [ ]:
# === Avaliação do agrupamento / Cluster evaluation ===
# ARI e NMI
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
ari = adjusted_rand_score(classes, clusters)
nmi = normalized_mutual_info_score(classes, clusters)
print(f"Adjusted Rand Index (ARI): {ari:.4f}")
print(f"Normalized Mutual Information (NMI): {nmi:.4f}")

Para as variáveis observáveis / For observable variables

In [ ]:
print("Colunas restantes no DataFrame dados:")
print(dados.head())

In [ ]:
# === Remover a primeira coluna de classes do DataFrame ===
dados = dados.iloc[:, 1:]

# === Transposta do DataFrame / Transpose of the DataFrame ===
dados_transposto = dados.T
display(dados_transposto.head())

In [ ]:
# === K-means ===
np.random.seed(1234)
kmeans = KMeans(n_clusters=3, n_init='auto', random_state=1234) # n_clusters = number of classes
clusters = kmeans.fit_predict(dados_transposto)

In [ ]:
# === PCA para reduzir a 2D para visualização ===
# Usamos o dados_transposto atual sem realizar nenhuma nova fatiamento ou exclusão
pca = PCA(n_components=2)
dados_pca = pca.fit_transform(dados_transposto)

# === Gráfico dos clusters ===
plt.figure(figsize=(10, 8))
scatter = plt.scatter(dados_pca[:, 0], dados_pca[:, 1], c=clusters, cmap="tab10", s=100, alpha=0.8)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-means Clustering (Atributos) com Identificação")

# Adicionar o nome de cada atributo (linha do transposto) ao lado de seu ponto correspondente
for i, nome_amostra in enumerate(dados_transposto.index):
    plt.annotate(nome_amostra, (dados_pca[i, 0], dados_pca[i, 1]), textcoords="offset points", xytext=(5,5), ha='left', fontsize=9, fontweight='bold')

# Criando a legenda para os clusters
handles, labels = scatter.legend_elements()
plt.legend(handles, [f'Cluster {i}' for i in range(len(handles))], title="Clusters", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()